# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pankaj1281/flyrank_ml_internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [19]:
import os

REPO_URL = "https://github.com/pankaj1281/flyrank_ml_internship.git"
REPO_PATH = "/content/flyrank_ml_internship"

if not os.path.exists(REPO_PATH):
    !git clone {REPO_URL}

print("Repository ready.")

Repository ready.


In [20]:
import pandas as pd
import numpy as np

DATA_PATH = "/content/flyrank_ml_internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

Dataset shape: (30000, 44)


## 1. Build the feature vector

I use content, search-performance, age, and freshness signals that are available before the prediction decision.

Numeric features are imputed with the median. Categorical features are imputed with the most frequent value and one-hot encoded.

The label is kept separate from the feature vector. I do not use trend_direction or trend_pct as model features because they are used to define the outcome.

In [21]:
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Label counts:")
print(df["is_declining_label"].value_counts())

Label counts:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


In [22]:
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

In [23]:
categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier"
]

feature_cols = numeric_features + categorical_features

print("Total feature columns:", len(feature_cols))

Total feature columns: 36


In [24]:
X = df[feature_cols].copy()
y = df["is_declining_label"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (30000, 36)
y shape: (30000,)


In [25]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_transformer = SimpleImputer(
    strategy="median"
)

categorical_transformer = SimpleImputer(
    strategy="most_frequent"
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

X_processed = preprocessor.fit_transform(X)

print("Processed feature shape:", X_processed.shape)

Processed feature shape: (30000, 36)


## 2. Feature notes

- Search and performance features describe previously observed page performance.
- Content age and days since last update describe freshness.
- Content type and main intent are categorical page attributes.
- Missing numeric values are filled with the median.
- Missing categorical values are filled with the most frequent category.
- These features are intended to be available before the prediction decision.
- IDs and label-derived fields are not used as model features.

In [26]:
missing = X.isna().mean().sort_values(ascending=False)

print("Missing values (%):")
print((missing * 100).round(2))


Missing values (%):
word_count                25.66
char_count_tier           25.66
word_count_tier           25.66
char_count                25.66
competition_level          8.70
cpc                        8.23
search_volume              8.23
competition                8.23
main_intent                7.91
scroll_rate                0.42
clicks_90d                 0.00
impressions_90d            0.00
days_with_impressions      0.00
days_with_sessions         0.00
impressions_last_30d       0.00
pageviews_90d              0.00
sessions_90d               0.00
users_90d                  0.00
engaged_sessions_90d       0.00
ai_sessions_90d            0.00
sessions_prev_30d          0.00
clicks_prev_30d            0.00
impressions_prev_30d       0.00
sessions_last_30d          0.00
clicks_last_30d            0.00
avg_position               0.00
content_age_days           0.00
days_since_last_update     0.00
ai_traffic_pct             0.00
engagement_rate            0.00
ctr                 

In [27]:
print("Categorical feature information:")

for col in categorical_features:
    print(
        col,
        "| unique values:",
        df[col].nunique(),
        "| missing:",
        df[col].isna().sum()
    )

Categorical feature information:
competition_level | unique values: 3 | missing: 2610
content_type | unique values: 3 | missing: 0
main_intent | unique values: 4 | missing: 2374
age_tier | unique values: 4 | missing: 0
freshness_tier | unique values: 4 | missing: 0
word_count_tier | unique values: 4 | missing: 7699
char_count_tier | unique values: 4 | missing: 7699
impression_tier | unique values: 4 | missing: 0
position_tier | unique values: 5 | missing: 0


## 3. The leakage hunt

I checked the feature vector for label-derived fields, future information, identifiers, and product decision fields.

`trend_direction` and `trend_pct` are excluded because they are related to the trend outcome used to define the label.

Future-window measurements are also excluded because they would not be available at the prediction moment.

The final feature vector contains only intended pre-decision information.

In [28]:
leakage_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

for col in leakage_columns:
    print(
        col,
        "→",
        "USED AS FEATURE" if col in feature_cols
        else "NOT USED AS FEATURE"
    )


trend_direction → NOT USED AS FEATURE
trend_pct → NOT USED AS FEATURE
is_declining_label → NOT USED AS FEATURE


In [29]:
id_columns = [
    "content_id",
    "client_id"
]

for col in id_columns:
    print(
        col,
        "→",
        "USED AS FEATURE" if col in feature_cols
        else "NOT USED AS FEATURE"
    )

content_id → NOT USED AS FEATURE
client_id → NOT USED AS FEATURE


## 4. What I excluded and why

- `content_id` — identifier only, not a predictive feature.
- `client_id` — used for grouping or validation, not prediction.
- `trend_direction` — related to the label and would cause leakage.
- `trend_pct` — trend-derived information and therefore excluded.
- `is_declining_label` — the target, not a feature.
- Future-window metrics — unavailable at the prediction moment.
- Product decision flags — represent decisions rather than predictive evidence.

In [30]:
excluded_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

bad_features = set(excluded_columns).intersection(feature_cols)

print("Excluded columns:")
for col in excluded_columns:
    print("-", col)

print("\nFinal leakage check:")

if len(bad_features) == 0:
    print("PASS — excluded fields are not model features.")
else:
    print("FAIL — these excluded fields are features:", bad_features)


Excluded columns:
- content_id
- client_id
- trend_direction
- trend_pct
- is_declining_label

Final leakage check:
PASS — excluded fields are not model features.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.